# Etapa 3: pipeline clássica de processamento de imagens

Projeto Contagem de Objetos em Imagens Industriais (`parafuso`, `porca`, `arruela`).

Este notebook mostra os resultados, as figuras e as métricas da pipeline clássica. A calibração (em train e valid) e a inferência final (no test) rodam em `scripts/classico.py`, e a avaliação roda em `scripts/avaliar.py`. As células abaixo leem o que esses scripts gravaram em `resultados/`. Só a última célula executa a pipeline, e apenas sobre uma imagem.

### Estrutura da pipeline (`scripts/classico.py`)
1. `etapa1_preprocessar`: escala de cinza, detecção de padding de rotação, filtro de mediana, correção de iluminação (`nenhum`, `gauss_divisao`, `morf_abertura`, `tophat`, `blackhat`) e CLAHE opcional.
2. `etapa2_segmentar`: detecção automática da polaridade pela mediana da faixa de borda válida, seguida de Otsu ou limiarização adaptativa.
3. `etapa3_morfologia`: abertura e fechamento configuráveis. Mantém duas máscaras, uma com furos preenchidos (limitados por uma área máxima relativa) e outra sem preenchimento.
4. `etapa4_separar_watershed`: transformada de distância e watershed. Para não dividir parafusos alongados em excesso, a suavização é proporcional ao raio típico, a altura mínima é global e local, e a distância mínima entre picos é relativa à largura típica.
5. `etapa5_componentes`: `connectedComponentsWithStats` com área relativa mínima e máxima, e extração dos descritores de forma (`razao_aspecto`, `proporcao_furo`, `circularidade`, `solidez`, `n_vertices`, `area_rel`).
6. `etapa6_classificar_componentes`: regra legível por limiares, derivados das distribuições do train (componentes com `IoU >= 0,5` contra o GT).
7. `etapa7_contar_por_classe`: soma os componentes de cada classe e devolve a contagem.


### Célula 1 — Ambiente e verificação do config

Importa `classico` e `avaliar` e confere o SHA-256 de `config_final.json` contra o `.sha256` registrado. O notebook só segue se a configuração congelada for a mesma da calibração.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

RAIZ = Path.cwd().resolve()
if not (RAIZ / 'scripts' / 'classico.py').exists():
    RAIZ = RAIZ.parent

sys.path.insert(0, str(RAIZ / 'scripts'))
import classico as cl
import avaliar as av

PASTA_RES = RAIZ / 'resultados' / 'classico'
PASTA_AV = RAIZ / 'resultados' / 'avaliacao'

cfg_path = PASTA_RES / 'config_final.json'
sha_registrado = (PASTA_RES / 'config_final.sha256').read_text(encoding='utf-8').split()[0]
sha_atual = cl.sha256_arquivo(cfg_path)
assert sha_registrado == sha_atual, 'Divergência no SHA-256 de config_final.json!'

cfg_final = json.loads(cfg_path.read_text(encoding='utf-8'))
print('Pipeline importada de:', cl.__file__)
print('Configuração congelada em:', cfg_final['gerado_em_utc'])
print('SHA-256 verificado:', sha_atual)

### Célula 2 — Configurações calibradas por dataset

Monta uma tabela com os parâmetros principais de `config_final.json` para cada dataset. A calibração usou só train e valid, com uma busca em grade de 32 a 48 combinações por dataset.


In [ ]:
campos_resumo = [
    'mediana_k', 'iluminacao_metodo', 'usar_clahe', 'polaridade',
    'seg_metodo', 'seg_bloco', 'seg_c',
    'morf_abertura_k', 'morf_fechamento_k', 'furo_max_area_rel',
    'ws_habilitado', 'ws_suavizacao_fator', 'ws_altura_rel', 'ws_dist_min_rel',
    'area_min_rel', 'area_max_rel'
]
linhas_cfg = []
for ds_id, ds_cfg in cfg_final['datasets'].items():
    row = {'dataset': ds_id, 'classes': ', '.join(cl.DATASETS_CLASSES[ds_id])}
    for c in campos_resumo:
        row[c] = ds_cfg[c]
    linhas_cfg.append(row)

display(pd.DataFrame(linhas_cfg))

### Célula 3 — Descritores no train e limiares de classificação

Mostra as estatísticas e os histogramas dos seis descritores de forma dos componentes casados com o GT no train (`IoU >= 0,5`). Essas distribuições justificam os limiares da etapa 6.


In [ ]:
for ds_id in ('nutsnbolts', 'screws_and_nuts'):
    display(Markdown(f'### Limiares e estatísticas — `{ds_id}`'))
    est = pd.DataFrame(cfg_final['estatisticas_descritores_train'][ds_id]).T
    display(est)
    hist_img = PASTA_RES / 'histogramas' / f'histogramas_{ds_id}_train.png'
    if hist_img.exists():
        display(Image(filename=str(hist_img), width=1000))

hist_geral = PASTA_RES / 'histogramas' / 'histogramas_geral_train.png'
if hist_geral.exists():
    display(Markdown('### Distribuição agregada (todos os datasets — `train`)'))
    display(Image(filename=str(hist_geral), width=1000))

### Célula 4 — Métricas no valid e no test

Lê os `metricas.csv` gerados por `scripts/avaliar.py`, que faz o casamento estrito 1:1 com `data_final/contagens_gt.csv`, e os mostra em formato largo. O valid é a base da calibração e o test é a avaliação final, com a configuração já congelada.


In [ ]:
def tabela_larga(caminho_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(caminho_csv)
    piv = df.pivot(index=['escopo', 'classe', 'dataset'], columns='metrica', values='valor').reset_index()
    cols = [
        'escopo', 'classe', 'dataset', 'n_imagens', 'soma_gt', 'soma_prevista',
        'mae', 'rmse', 'vies', 'acerto_exato_pct', 'acerto_tol1_pct'
    ]
    ordem_escopo = {'geral': 0, 'classe': 1, 'classe_dataset': 2}
    piv['_ord'] = piv['escopo'].map(ordem_escopo)
    piv = piv.sort_values(['_ord', 'classe', 'dataset']).drop(columns=['_ord'])
    return piv[cols].round(2)

display(Markdown('### Split `valid` (calibração)'))
display(tabela_larga(PASTA_AV / 'classico_valid' / 'metricas.csv'))

display(Markdown('### Split `test` (avaliação final congelada)'))
display(tabela_larga(PASTA_AV / 'classico_test' / 'metricas.csv'))

### Célula 5 — Diagnóstico de segmentação e de classificação

Compara o total de componentes (sem classe) com a soma do GT por imagem, para separar falhas de segmentação e de watershed das trocas puras de classe. Lista também as 10 piores imagens do test.


In [ ]:
df_tot = pd.read_csv(PASTA_RES / 'total_por_imagem.csv')
df_pi = pd.read_csv(PASTA_AV / 'classico_test' / 'por_imagem.csv')
soma_cls = df_pi.groupby(['dataset', 'imagem'])['erro_abs'].sum().reset_index(name='soma_abs_classes')
m = df_tot.merge(soma_cls, on=['dataset', 'imagem'])
m['trocas_classe_puras'] = (m['soma_abs_classes'] - m['erro_abs_total']) / 2.0

resumo_diag = m.groupby('dataset').agg(
    n_imagens=('imagem', 'count'),
    soma_gt=('total_gt', 'sum'),
    soma_previsto=('total_previsto', 'sum'),
    mae_total_seg=('erro_abs_total', 'mean'),
    vies_total=('erro_total', 'mean'),
    exato_total_pct=('erro_abs_total', lambda s: (s == 0).mean() * 100.0),
    tol1_total_pct=('erro_abs_total', lambda s: (s <= 1).mean() * 100.0),
    trocas_classe_med=('trocas_classe_puras', 'mean'),
).round(2)
display(resumo_diag)

display(Markdown('### 10 piores imagens no `test` (`por_imagem.csv`)'))
top10 = df_pi.sort_values(['erro_abs', 'dataset', 'imagem'], ascending=[False, True, True]).head(10)
display(top10[['dataset', 'imagem', 'classe', 'contagem_gt', 'contagem_prevista', 'erro', 'erro_abs']])

### Célula 6 — Figuras das etapas da pipeline

Exibe as figuras de `resultados/classico/figuras/`, com 2 imagens do test por dataset. Cada painel traz a imagem original, a pré-processada, a binária, a morfologia preenchida, a transformada de distância, os marcadores, o watershed e o resultado classificado.


In [ ]:
for fig_path in sorted((PASTA_RES / 'figuras').glob('*.png')):
    display(Markdown(f'#### `{fig_path.name}`'))
    display(Image(filename=str(fig_path), width=1050))

### Célula 7 — Demonstração da API

Chama as 7 etapas de `scripts/classico.py` em sequência sobre a primeira imagem do valid de `nutsnbolts`, para mostrar como as funções se encadeiam.


In [ ]:
import cv2

ds_demo = 'nutsnbolts'
img_demo = sorted((RAIZ / 'data_final' / 'base' / ds_demo / 'valid' / 'images').iterdir())[0]
bgr = cv2.imread(str(img_demo), cv2.IMREAD_COLOR)
cfg_demo = cfg_final['datasets'][ds_demo]

proc, pad = cl.etapa1_preprocessar(bgr, cfg_demo)
binaria, pol = cl.etapa2_segmentar(proc, pad, cfg_demo)
m_preench, m_sem = cl.etapa3_morfologia(binaria, cfg_demo)
rot_ws, dist_s, marcs = cl.etapa4_separar_watershed(m_preench, cfg_demo)
comps = cl.etapa5_componentes(rot_ws, m_sem, cfg_demo)
classif = cl.etapa6_classificar_componentes(comps, cl.DATASETS_CLASSES[ds_demo], cfg_demo)
cont = cl.etapa7_contar_por_classe(classif, cl.DATASETS_CLASSES[ds_demo])

print('Imagem:', img_demo.name)
print('Polaridade detectada:', pol)
print('Componentes válidos:', len(comps))
print('Contagem por classe:', cont)